In [1]:
# === SESSION START: run this first, every time ===
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO_DIR = "/content/drive/MyDrive/projects/namibia-baseline-pack-demo"
%cd "{REPO_DIR}"

import sys, importlib
sys.path.insert(0, f"{REPO_DIR}/src")

import ee, geemap
try:
    ee.Initialize(project=userdata.get("EE_PROJECT"))
except Exception:
    ee.Authenticate()
    ee.Initialize(project=userdata.get("EE_PROJECT"))
print("EE ready:", ee.String("ok").getInfo())

# --- state check: what is actually in the repo right now ---
!find . -path ./.git -prune -o -path ./outputs -prune -o -print | sort
!ls -lh outputs/rasters outputs/figures outputs/tables 2>/dev/null
!git status --short && git log --oneline -3

Mounted at /content/drive
/content/drive/MyDrive/projects/namibia-baseline-pack-demo
EE ready: ok
.
./config.yaml
./CREDITS.md
./data
./data/vector
./data/vector/aoi.gpkg
./data/vector/training.gpkg
./docs
./docs/class_key.md
./docs/time_log.md
./.gitignore
./LICENSE
./notebooks
./notebooks/05_rf_spectral.ipynb
./README.md
./requirements.txt
./src
./src/baseline_pack
./src/baseline_pack/aoi.py
./src/baseline_pack/composites.py
./src/baseline_pack/gitutils.py
./src/baseline_pack/__init__.py
./src/baseline_pack/__pycache__
./src/baseline_pack/__pycache__/aoi.cpython-313.pyc
./src/baseline_pack/__pycache__/composites.cpython-313.pyc
./src/baseline_pack/__pycache__/gitutils.cpython-313.pyc
./src/baseline_pack/__pycache__/__init__.cpython-313.pyc
./src/baseline_pack/__pycache__/trend.cpython-313.pyc
./src/baseline_pack/trend.py
outputs/figures:
total 214K
-rw------- 1 root root 214K Oct 21 09:58 ndvi_rain_2016_2025.png

outputs/rasters:
total 33M
-rw------- 1 root root 2.2M Oct 21 10:01 dry

In [2]:
from pathlib import Path
Path("notebooks").mkdir(exist_ok=True)
Path("notebooks/.gitkeep").touch()
!ls -la notebooks

total 10
drwx------ 2 root root 4096 Oct 22 07:29 .
drwx------ 8 root root 4096 Oct 21 07:10 ..
-rw------- 1 root root 1647 Oct 22 07:29 05_rf_spectral.ipynb
-rw------- 1 root root    0 Oct 22 07:29 .gitkeep


In [3]:
!git show --stat --oneline c93b76a

import subprocess, geopandas as gpd
def count_polys(path):
    try:
        return len(gpd.read_file(path, layer="training"))
    except Exception as e:
        return f"not readable ({type(e).__name__})"

for ref in ("origin/main", "HEAD"):
    blob = subprocess.run(["git", "show", f"{ref}:data/vector/training.gpkg"], capture_output=True).stdout
    tmp = f"/tmp/{ref.replace('/', '_')}.gpkg"
    open(tmp, "wb").write(blob)
    print(f"{ref:12s} →", count_polys(tmp) if blob else "file not in this version")
print(f"{'Drive copy':12s} →", count_polys("data/vector/training.gpkg"))

c93b76a (origin/main, origin/HEAD) Add files via upload
 data/vector/training.gpkg | Bin 0 -> 135168 bytes
 1 file changed, 0 insertions(+), 0 deletions(-)
origin/main  → 98
HEAD         → 100
Drive copy   → 100


In [4]:
from baseline_pack.gitutils import git_push
git_push("Training polygons (100) and Session 5 notebook")
!git log --oneline -3

git add → 
git commit → [main 4f2cc81] Training polygons (100) and Session 5 notebook
 2 files changed, 1 insertion(+)
 create mode 100644 notebooks/.gitkeep
 create mode 100644 notebooks/05_rf_spectral.ipynb
git push → warning: redirecting to https://github.com/ybrosig-geo/namibia-baseline-pack-demo.git/
To https://github.com/ybrosig-geo/namibia-baseline-pack-demo.git
   2c49d21..4f2cc81  HEAD -> main
4f2cc81 (HEAD -> main) Training polygons (100) and Session 5 notebook
2c49d21 Add training polygons (5-class key)
c93b76a (origin/main, origin/HEAD) Add files via upload


In [7]:
import yaml, geopandas as gpd, pandas as pd
import baseline_pack.classify as cl; importlib.reload(cl)
from baseline_pack.aoi import gdf_to_ee
from baseline_pack.composites import seasonal_composite, terrain

cfg = yaml.safe_load(open("config.yaml"))
NAMES = {int(k): v["name"] for k, v in cfg["classes"].items()}
PALETTE = [cfg["classes"][k]["color"].lstrip("#") for k in sorted(cfg["classes"])]
CLASSES = sorted(NAMES)

aoi = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi")).geometry()
region = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi_buffer")).geometry()
dry, _, _ = seasonal_composite(region, cfg["current_year"], cfg["dry_season_months"], "dry")
wet, _, _ = seasonal_composite(region, cfg["current_year"], cfg["wet_season_months"], "wet")
stack = cl.feature_stack(dry, wet, terrain(region))
FEATURES = stack.bandNames().getInfo()

train_gdf = cl.prepare_training(gpd.read_file("data/vector/training.gpkg", layer="training"), inner_buffer_m=5)
pts, balanced = cl.sample_pixels(stack, train_gdf)
print(len(FEATURES), "features;", len(train_gdf), "polygons left after 10 m inward buffer")
display(cl.pixel_counts(pts, CLASSES))

/content/drive/MyDrive/projects/namibia-baseline-pack-demo/src/baseline_pack/classify.py:21: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  g = g[g.geometry.notna() & ~g.geometry.is_empty].copy()


31 features; 98 polygons left after 10 m inward buffer


,train_px,test_px
class_id,,
1,108,25
2,247,73
3,254,72
4,488,222
5,412,76


In [8]:
import yaml
from pathlib import Path
cfg = yaml.safe_load(open("config.yaml"))
cfg["classes"][1]["name"] = "Closed woody vegetation"
cfg["classes"][2]["name"] = "Bush savanna"
yaml.safe_dump(cfg, open("config.yaml", "w"), sort_keys=False)

p = Path("docs/class_key.md")
t = p.read_text()
t = t.replace("| 1 | Dense bush | Woody canopy >= 40%; crowns mostly touching. Includes scattered taller trees. |",
              "| 1 | Closed woody vegetation | Woody canopy >= 70%; crowns nearly continuous, little or no ground visible. Thickets, hill slopes, drainage lines; often includes taller trees. |")
t = t.replace("| 2 | Open bush / shrub savanna | Woody canopy 10-40%; separate shrubs/trees with grass between. |",
              "| 2 | Bush savanna | Woody canopy 10-70%; distinct shrubs/trees with ground or grass visible between them. Most encroached rangeland. |")
t += "\n## Change log\n- 2026-10-22: Class 1 threshold raised from 40% to 70% (closed woody vegetation) and class 2 widened to 10-70%, to match how training areas were actually interpreted.\n"
p.write_text(t)
print(p.read_text())

# Land-cover class key (reference year 2025)

Woody canopy cover = share of ground covered by tree and shrub crowns seen from above,
judged visually on high-resolution imagery. Height is not distinguished.

| ID | Class | Definition |
|---|---|---|
| 1 | Closed woody vegetation | Woody canopy >= 70%; crowns nearly continuous, little or no ground visible. Thickets, hill slopes, drainage lines; often includes taller trees. |
| 2 | Bush savanna | Woody canopy 10-70%; distinct shrubs/trees with ground or grass visible between them. Most encroached rangeland. |
| 3 | Grassland / open | Woody canopy < 10%; includes cleared blocks now dominated by grass. |
| 4 | Bare / disturbed ground | Mostly exposed soil or rock: wide gravel tracks, quarries, fresh clearing, construction, dry pan floors. |
| 5 | Built-up | Buildings, paved surfaces and yards. |

## Decision rules for edge cases
- Scattered tall trees in grass: count their crowns; if total canopy < 10% -> Grassland / open.
- Cleared block w

In [9]:
!ls -la data/vector/
t = gpd.read_file("data/vector/training.gpkg", layer="training")
notes = t["notes"].fillna("")
print("Polygons:", len(t), "| new:", notes.str.contains("added 2026-10-22").sum(),
      "| reassigned:", notes.str.contains("reassigned").sum())
print(t.groupby("class_id").size().rename("polygons"))

total 260
drwx------ 2 root root   4096 Oct 21 07:11 .
drwx------ 3 root root   4096 Oct 21 07:11 ..
-rw------- 1 root root 118784 Oct 21 07:42 aoi.gpkg
-rw------- 1 root root 139264 Oct 22 08:06 training.gpkg
Polygons: 107 | new: 0 | reassigned: 0
class_id
1    26
2    28
3    19
4    17
5    17
Name: polygons, dtype: int64


In [10]:
import yaml
from pathlib import Path
cfg = yaml.safe_load(open("config.yaml"))
cfg["classes"][1]["name"] = "Closed woody vegetation"
cfg["classes"][2]["name"] = "Bush savanna"
yaml.safe_dump(cfg, open("config.yaml", "w"), sort_keys=False)

p = Path("docs/class_key.md")
t = p.read_text()
t = t.replace("| 1 | Dense bush | Woody canopy >= 40%; crowns mostly touching. Includes scattered taller trees. |",
              "| 1 | Closed woody vegetation | Woody canopy >= 70%; crowns nearly continuous, little or no ground visible. Thickets, hill slopes, drainage lines; often includes taller trees. |")
t = t.replace("| 2 | Open bush / shrub savanna | Woody canopy 10-40%; separate shrubs/trees with grass between. |",
              "| 2 | Bush savanna | Woody canopy 10-70%; distinct shrubs/trees with ground or grass visible between them. Most encroached rangeland. |")
t += "\n## Change log\n- 2026-10-22: Class 1 threshold raised from 40% to 70% (closed woody vegetation) and class 2 widened to 10-70%, to match how training areas were actually interpreted.\n"
p.write_text(t)
print(p.read_text())

# Land-cover class key (reference year 2025)

Woody canopy cover = share of ground covered by tree and shrub crowns seen from above,
judged visually on high-resolution imagery. Height is not distinguished.

| ID | Class | Definition |
|---|---|---|
| 1 | Closed woody vegetation | Woody canopy >= 70%; crowns nearly continuous, little or no ground visible. Thickets, hill slopes, drainage lines; often includes taller trees. |
| 2 | Bush savanna | Woody canopy 10-70%; distinct shrubs/trees with ground or grass visible between them. Most encroached rangeland. |
| 3 | Grassland / open | Woody canopy < 10%; includes cleared blocks now dominated by grass. |
| 4 | Bare / disturbed ground | Mostly exposed soil or rock: wide gravel tracks, quarries, fresh clearing, construction, dry pan floors. |
| 5 | Built-up | Buildings, paved surfaces and yards. |

## Decision rules for edge cases
- Scattered tall trees in grass: count their crowns; if total canopy < 10% -> Grassland / open.
- Cleared block w

In [14]:
import yaml, geopandas as gpd, pandas as pd
import baseline_pack.classify as cl; importlib.reload(cl)
from baseline_pack.aoi import gdf_to_ee
from baseline_pack.composites import seasonal_composite, terrain

cfg = yaml.safe_load(open("config.yaml"))
NAMES = {int(k): v["name"] for k, v in cfg["classes"].items()}
PALETTE = [cfg["classes"][k]["color"].lstrip("#") for k in sorted(cfg["classes"])]
CLASSES = sorted(NAMES)

aoi = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi")).geometry()
region = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi_buffer")).geometry()
dry, _, _ = seasonal_composite(region, cfg["current_year"], cfg["dry_season_months"], "dry")
wet, _, _ = seasonal_composite(region, cfg["current_year"], cfg["wet_season_months"], "wet")
stack = cl.feature_stack(dry, wet, terrain(region))
FEATURES = stack.bandNames().getInfo()

train_gdf = cl.prepare_training(gpd.read_file("data/vector/training.gpkg", layer="training"), inner_buffer_m=5)
pts, balanced = cl.sample_pixels(stack, train_gdf)
print(len(FEATURES), "features;", len(train_gdf), "polygons left after 5 m inward buffer")
display(cl.pixel_counts(pts, CLASSES))

/content/drive/MyDrive/projects/namibia-baseline-pack-demo/src/baseline_pack/classify.py:22: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  g["class_id"] = g["class_id"].astype(int)


31 features; 105 polygons left after 5 m inward buffer


,train_px,test_px
class_id,,
1,222,95
2,242,126
3,157,169
4,484,226
5,319,169


In [15]:
import inspect
print("Area-based split active:", "held" in inspect.getsource(cl.prepare_training))
share = train_gdf.assign(a=train_gdf.area).groupby(["class_id", "split"])["a"].sum().unstack()
print((100 * share["test"] / share.sum(axis=1)).round(0).rename("test_area_%"))

Area-based split active: True
class_id
1    30.0
2    34.0
3    52.0
4    31.0
5    35.0
Name: test_area_%, dtype: float64
